# Análisis y pronóstico de BTC — función no lineal + Modelo Ventana Móvil
### (versión funcional — Programación Funcional, Unidad I-III)

**Dataset:** `btc-timeseries.json` — 365 observaciones diarias de BTC (precio = (máx+mín) del día / 2), sin gaps de fechas.

**Objetivo:** construir dos modelos de pronóstico que **no** sean una recta ni un polinomio:

1. **Ajuste por función lambda** — tangente hiperbólica + armónicos de Fourier, ajustada por mínimos cuadrados no lineales (`scipy.optimize.curve_fit`). Capaz de **interpolar** y **extrapolar**.
2. **Modelo Ventana Móvil** (Media Móvil de 7 días) con *features* de rezagos y estadísticos móviles.

Ambos se evalúan con **walk-forward validation**, comparados contra un **baseline ingenuo**.

> **Nota metodológica.** Esta versión reescribe el pipeline aplicando los conceptos de la cátedra vistos hasta la Clase 4:
> funciones puras y transparencia referencial (Clase 4), inmutabilidad vía `dataclass(frozen=True)`/`NamedTuple`/tuplas
> (Clase 4), funciones de orden superior — fábricas de funciones y closures (Clases 3-4), el patrón **Functor**
> (`.map()` que preserva la forma, con sus dos leyes verificadas por asserts) y el patrón **Monoid** (acumular resultados
> de folds con `reduce`, Clase 4). El manejo de fits que pueden fallar usa un tipo `FitOutcome` (éxito/error) que
> **anticipa** el patrón Mónada (Maybe/Either) — sin implementarlo formalmente, porque eso se formaliza recién en la
> Clase 8. La función de transición pura de la Máquina de Turing (Clase 2, `δ`) inspira el diseño de extrapolación recursiva
> puro de la extrapolación recursiva.

In [ ]:
import json
from dataclasses import dataclass
from functools import reduce
from typing import Callable, NamedTuple, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import wilcoxon
from statsmodels.tsa.stattools import adfuller, acf

plt.rcParams["figure.figsize"] = (11, 4.5)
np.random.seed(0)

# --- Configuración inmutable (Clase 4: @dataclass(frozen=True)) ---
# En vez de constantes sueltas, agrupamos cada configuración en un registro
# congelado: no se puede mutar por accidente en ningún lugar del notebook.

@dataclass(frozen=True)
class WFConfig:
    horizon: int = 7      # días pronosticados por ventana
    step: int = 7         # desplazamiento entre folds
    min_train: int = 150  # tamaño mínimo de entrenamiento del primer fold

@dataclass(frozen=True)
class LambdaConfig:
    harmonics: int = 4
    periods_u: Tuple[float, ...] = (1.0, 0.5, 1/3, 0.25, 2.0)

WF = WFConfig()
LCFG = LambdaConfig()
LAGS: Tuple[int, ...] = (1, 2, 3, 7, 14, 30)
WINDOWS: Tuple[int, ...] = (7, 14, 30)
def mean_absolute_error(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))



## 0. Calentamiento: Functor sobre listas (ejercicio de Clase 4)

Antes de meternos en el pipeline, un chequeo directo de lo pedido en la práctica de Clase 4:
implementar un Functor y **verificar sus dos leyes con `assert`, no con la intuición**.

- **Ley de identidad:** mapear la identidad no cambia nada.
- **Ley de composición:** mapear `g` y después `h` da lo mismo que mapear `h∘g` de una sola vez.

`map` sobre listas ya es un Functor en Python — lo dejamos explícito antes de reusar la misma idea
más abajo con `FitOutcome.map(...)`.

In [ ]:
def fmap(f: Callable, xs: list) -> list:
    return list(map(f, xs))

xs_demo = [1, 2, 3, 4]
identidad = lambda x: x
g = lambda x: x + 1
h = lambda x: x * 2

assert fmap(identidad, xs_demo) == xs_demo, "ley de identidad falló"
assert fmap(lambda x: h(g(x)), xs_demo) == fmap(h, fmap(g, xs_demo)), "ley de composición falló"
print("Leyes de Functor verificadas sobre listas (identidad y composición). ✓")

## 1. Carga y exploración de datos (EDA)

In [ ]:
from pathlib import Path
from urllib.request import urlopen

DATA_PATH = Path("btc-timeseries.json")
DATA_URL = "https://raw.githubusercontent.com/number1angel/btc-funcional-analisis/main/btc-timeseries.json"

if DATA_PATH.exists():
    with DATA_PATH.open(encoding="utf-8") as f:
        data = json.load(f)
else:
    with urlopen(DATA_URL) as response:
        data = json.load(response)

print("Nota del dataset:", data["nota"])

df = pd.DataFrame(data["obs"])
df["fecha"] = pd.to_datetime(df["fecha"])
n_all = len(df)

print(f"Observaciones: {n_all}")
print(f"Rango de fechas: {df['fecha'].min().date()} -> {df['fecha'].max().date()}")
print(df["precio"].describe())

In [ ]:
fig, ax = plt.subplots()
ax.plot(df["fecha"], df["precio"], lw=1.2, color="#f2a900")
ax.set_title("BTC — precio diario")
ax.set_ylabel("USD")
ax.grid(alpha=0.3)
plt.show()

### Estacionariedad, autocorrelación y volatilidad

Si el **precio** no es estacionario pero los **retornos logarítmicos** sí, estamos ante un
comportamiento tipo *random walk* — típico en criptoactivos, y la razón técnica de fondo por la
que ningún ajuste determinístico puede "predecir" el futuro con certeza.

In [ ]:
precio = df["precio"].values
log_ret = np.diff(np.log(precio))

adf_precio = adfuller(precio)
adf_ret = adfuller(log_ret)

print(f"ADF sobre el precio      -> stat={adf_precio[0]:.3f}  p-valor={adf_precio[1]:.4f}")
print(f"ADF sobre log-retornos   -> stat={adf_ret[0]:.3f}  p-valor={adf_ret[1]:.4f}")

vol_diaria = log_ret.std(ddof=1)
vol_anual = vol_diaria * np.sqrt(365)
print(f"\nVolatilidad diaria (std log-retornos): {vol_diaria:.4f}")
print(f"Volatilidad anualizada aproximada:      {vol_anual:.2%}")

acfs = acf(log_ret, nlags=10)
print("\nAutocorrelación de retornos (lags 1-10):", np.round(acfs[1:], 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df["fecha"].iloc[1:], log_ret, lw=0.8, color="#3b6ea5")
axes[0].set_title("Log-retornos diarios")
axes[0].grid(alpha=0.3)
axes[1].hist(log_ret, bins=40, color="#3b6ea5", alpha=0.8)
axes[1].set_title("Distribución de log-retornos")
plt.tight_layout()
plt.show()

**Lectura del diagnóstico:**
- El precio **no es estacionario** (p-valor alto) → una tendencia determinística simple
  (recta/polinomio) no tiene sustento estadístico.
- Los **retornos sí son estacionarios** (p-valor ≈ 0) → la estructura predecible, si existe,
  vive en los retornos de corto plazo, no en el nivel del precio.
- Autocorrelación lag-1 moderada → hay algo de *momentum* de un día para el otro, explotable
  por un modelo de Ventana Móvil con features de rezago.
- La volatilidad anualizada resume qué tan ruidosos son los tramos de la serie.

## 2. Metodología de evaluación: Walk-Forward Validation

Ventana expansiva: entrenar con `[0, start)`, pronosticar `horizon` días, comparar contra el
valor real, deslizar `start` y repetir. Los `folds` quedan como una **tupla inmutable** — nadie
en el resto del notebook puede mutarla por accidente.

In [ ]:
folds: Tuple[int, ...] = tuple(range(WF.min_train, n_all - WF.horizon + 1, WF.step))
print(f"Número de folds de walk-forward: {len(folds)}")

## 3. Modelo 1 — Ajuste por función lambda (no lineal, no polinomial)

$$f(u) = a_0 + a_1 \tanh\big(a_2 (u - a_3)\big) + \sum_{k=1}^{K} \big[b_k \sin(2\pi k u / P) + c_k \cos(2\pi k u / P)\big]$$

`u = t / n` es el tiempo normalizado. `make_model` es una **función de orden superior**: es una
fábrica que devuelve una función pura (`model`) — un *closure* sobre `period_u` y `harmonics`,
sin ningún estado mutable de por medio.

Para manejar el caso en que `curve_fit` no converge con **ningún** período candidato, en vez de
dejar que el notebook explote con un error críptico (como en la versión anterior) devolvemos un
`FitOutcome`: un registro inmutable de éxito/error con un método `.map()` que **preserva la
forma** — si hubo error, lo propaga sin tocarlo; si hubo éxito, transforma el valor. Es el mismo
patrón Functor de la sección 0, aplicado a un resultado que puede fallar. (Esto anticipa el
patrón **Mónada** — Maybe/Either — que se formaliza recién en la Clase 8; acá nos alcanza con la
mitad Functor.)

In [ ]:
class FitOutcome(NamedTuple):
    ok: bool
    value: Optional[Callable] = None
    rmse: Optional[float] = None
    error: Optional[str] = None

    def map(self, f: Callable) -> "FitOutcome":
        '''Functor: transforma `value` sólo si hubo éxito; si no, propaga el error intacto.'''
        return self if not self.ok else FitOutcome(ok=True, value=f(self.value), rmse=self.rmse)


def make_model(period_u: float, harmonics: int) -> Callable:
    '''HOF: fábrica de funciones puras. Cada llamada devuelve un closure nuevo,
    sin compartir estado con las demás.'''
    def model(u, a0, a1, a2, a3, *bc):
        val = a0 + a1 * np.tanh(a2 * (u - a3))
        for k in range(harmonics):
            b, c = bc[2 * k], bc[2 * k + 1]
            w = 2 * np.pi * (k + 1) / period_u
            val = val + b * np.sin(w * u) + c * np.cos(w * u)
        return val
    return model


def fit_lambda_function(t_train: np.ndarray, y_train: np.ndarray, cfg: LambdaConfig = LCFG) -> FitOutcome:
    '''Función pura: misma entrada, mismo resultado; no muta `t_train`/`y_train` ni nada externo.
    Prueba varios períodos candidatos y bounds razonables (evita que curve_fit se vaya a un
    óptimo local absurdo), y se queda con el de menor RMSE in-sample.'''
    n = len(t_train)
    u = t_train / n
    bounds_lo = [-np.inf, -np.inf, 1e-6, -5.0] + [-np.inf, -np.inf] * cfg.harmonics
    bounds_hi = [np.inf, np.inf, 50.0, 5.0] + [np.inf, np.inf] * cfg.harmonics

    def try_period(period_u):
        model = make_model(period_u, cfg.harmonics)
        p0 = [y_train.mean(), (y_train.max() - y_train.min()) / 2, 3, 0.5] + [1.0, 1.0] * cfg.harmonics
        try:
            popt, _ = curve_fit(model, u, y_train, p0=p0, bounds=(bounds_lo, bounds_hi), maxfev=40000)
            pred = model(u, *popt)
            rmse = np.sqrt(mean_squared_error(y_train, pred))
            return (rmse, model, popt)
        except Exception:
            return None

    candidatos = list(filter(None, map(try_period, cfg.periods_u)))
    if not candidatos:
        return FitOutcome(ok=False, error="curve_fit no convergió con ningún período candidato")

    rmse, model, popt = min(candidatos, key=lambda c: c[0])
    f = lambda t_new: model(np.asarray(t_new, dtype=float) / n, *popt)
    return FitOutcome(ok=True, value=f, rmse=rmse)

In [ ]:
t_all = df["t"].values.astype(float)
y_all = df["precio"].values.astype(float)

fit_in = fit_lambda_function(t_all, y_all)
assert fit_in.ok, fit_in.error
f_lambda = fit_in.value

pred_in_sample = f_lambda(t_all)
mae_in = mean_absolute_error(y_all, pred_in_sample)
rmse_in = np.sqrt(mean_squared_error(y_all, pred_in_sample))
mape_in = np.mean(np.abs((y_all - pred_in_sample) / y_all)) * 100
print(f"Ajuste in-sample -> MAE={mae_in:,.0f}  RMSE={rmse_in:,.0f}  MAPE={mape_in:.2f}%")

fig, ax = plt.subplots()
ax.plot(df["fecha"], y_all, label="Real", lw=1.2, color="#f2a900")
ax.plot(df["fecha"], pred_in_sample, label="Función lambda (ajuste)", lw=1.6, color="#2b2b2b", ls="--")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Ajuste por función lambda — interpolación dentro de la muestra")
plt.show()

### Chequeo honesto: ¿el error bajo "in-sample" se sostiene afuera?

Probamos prediciendo los últimos 30 días **sin haberlos visto** en el ajuste.

In [ ]:
HOLDOUT = 30
t_tr, y_tr = t_all[:-HOLDOUT], y_all[:-HOLDOUT]
t_te, y_te = t_all[-HOLDOUT:], y_all[-HOLDOUT:]

fit_holdout = fit_lambda_function(t_tr, y_tr)
assert fit_holdout.ok, fit_holdout.error
pred_te = fit_holdout.value(t_te)

mae_out = mean_absolute_error(y_te, pred_te)
rmse_out = np.sqrt(mean_squared_error(y_te, pred_te))
mape_out = np.mean(np.abs((y_te - pred_te) / y_te)) * 100
print(f"Extrapolación out-of-sample (30 días) -> MAE={mae_out:,.0f}  RMSE={rmse_out:,.0f}  MAPE={mape_out:.2f}%")
print(f"(vs. {mape_in:.2f}% de MAPE in-sample — la diferencia es la señal de alerta importante)")

fig, ax = plt.subplots()
ax.plot(df["fecha"].iloc[-60:], y_all[-60:], label="Real", lw=1.4, color="#f2a900")
ax.plot(df["fecha"].iloc[-HOLDOUT:], pred_te, label="Extrapolación función lambda", lw=1.6, color="#2b2b2b", ls="--")
ax.axvline(df["fecha"].iloc[-HOLDOUT], color="grey", ls=":", label="Inicio del holdout")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Función lambda: ¿cómo le va prediciendo 30 días 'a ciegas'?")
plt.show()

## 4. Modelo 2 — Modelo Ventana Móvil (Media Móvil de 7 días)

Pronóstico basado en promediar recursivamente los últimos 7 días con un enfoque puramente funcional.


In [ ]:
# Celda eliminada: ya no generamos features para ML


Se descartaron los hiperparámetros de ML para usar el modelo funcional.


In [ ]:
def mape(y_true, y_pred) -> float:
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)

# Modelo de Ventana Móvil (Media Móvil de 7 días) usando recursión/reduce funcional
# Toma un historial y predice n pasos promediando los últimos 7 días
predict_ventana_movil = lambda pasos, hist: reduce(
    lambda acc, _: acc + (sum(acc[-7:]) / 7,), 
    range(pasos), 
    tuple(hist)
)[-pasos:]



## 5. Walk-forward: comparación real entre Naive / Función lambda / Ventana Móvil

Cada fold produce un registro **inmutable** (`FoldResult`, un `NamedTuple`). En vez de un `for`
con `.append()` sobre una lista mutable, los folds se evalúan con `map` y se combinan con
`reduce` — el patrón **Monoid**: el elemento neutro es la tupla vacía `()` y la operación
asociativa es la concatenación de tuplas.

In [ ]:
class FoldResult(NamedTuple):
    start: int
    naive_mae: float
    naive_mape: float
    lambda_mae: float
    lambda_mape: float
    movil_mae: float
    movil_mape: float

def evaluate_fold(start: int) -> Optional[FoldResult]:
    t_train, y_train = t_all[:start], y_all[:start]
    t_test, y_test = t_all[start:start + WF.horizon], y_all[start:start + WF.horizon]
    naive_pred = np.repeat(y_train[-1], WF.horizon)

    fit = fit_lambda_function(t_train, y_train)
    if not fit.ok:
        return None
    lam_pred = fit.value(t_test)

    # Predicción Ventana Móvil 7 días
    movil_pred = predict_ventana_movil(WF.horizon, y_train)

    return FoldResult(
        start=start,
        naive_mae=mean_absolute_error(y_test, naive_pred), naive_mape=mape(y_test, naive_pred),
        lambda_mae=mean_absolute_error(y_test, lam_pred), lambda_mape=mape(y_test, lam_pred),
        movil_mae=mean_absolute_error(y_test, movil_pred), movil_mape=mape(y_test, movil_pred),
    )

fold_results: Tuple[FoldResult, ...] = reduce(
    lambda acc, r: acc + (r,),
    filter(lambda r: r is not None, map(evaluate_fold, folds)),
    (),
)
print(f"Folds evaluados con éxito: {len(fold_results)} / {len(folds)}")

res = pd.DataFrame(fold_results)
res.tail()



In [ ]:
tabla = pd.DataFrame({
    "MAE promedio (USD)": [res.naive_mae.mean(), res.lambda_mae.mean(), res.movil_mae.mean()],
    "MAPE promedio (%)": [res.naive_mape.mean(), res.lambda_mape.mean(), res.movil_mape.mean()],
}, index=["Naive (baseline)", "Función lambda", "Ventana Móvil (7 días)"])
print(tabla.round(2))

fig, ax = plt.subplots()
tabla["MAPE promedio (%)"].plot(kind="bar", ax=ax, color=["#999999", "#2b2b2b", "#3b6ea5"])
ax.set_title(f"MAPE promedio en walk-forward ({len(fold_results)} ventanas, horizonte {WF.horizon} días)")
ax.set_ylabel("MAPE (%)")
plt.xticks(rotation=0)
plt.show()



### ¿La diferencia es real o es ruido de muestreo?

Comparar promedios no alcanza para decir que un modelo "le gana" a otro — con folds pareados
(mismos puntos de corte) corresponde un test no paramétrico pareado. Usamos
**Wilcoxon signed-rank** sobre el MAPE de cada fold: `naive vs. lambda` y `naive vs. Ventana Móvil`.

In [ ]:
stat_lambda, p_lambda = wilcoxon(res["naive_mape"], res["lambda_mape"])
stat_movil, p_movil = wilcoxon(res["naive_mape"], res["movil_mape"])

print(f"Wilcoxon (naive vs. función lambda) -> p-valor = {p_lambda:.4f}")
print(f"Wilcoxon (naive vs. Ventana Móvil)  -> p-valor = {p_movil:.4f}")
print("\n(p < 0.05 sugiere que la diferencia frente al naive no es casualidad de muestreo;)")



**Interpretación honesta:**
- Si el **Modelo Ventana Móvil** queda por debajo del naive *y* el test de Wilcoxon da un p-valor
  bajo, hay evidencia de señal real de corto plazo, no sólo ruido.
- Si la **función lambda** queda por encima del naive, es la confirmación esperable: una curva
  ajustada a toda la forma histórica no necesariamente predice bien los próximos días, porque su
  "giro" depende de datos que en producción todavía no existen. Su valor está más en
  **interpolar y describir la forma general** que en pronosticar con precisión de corto plazo.
- El baseline naive es difícil de vencer en un activo con comportamiento de paseo aleatorio — y
  eso en sí mismo es información valiosa.

## 6. Extrapolación final a 30 días

Entrenamos ambos modelos con **todo** el dataset y proyectamos 30 días. Para el Ventana Móvil, el
pronóstico es **recursivo**. En vez de un `for` que hace `.append()` sobre listas mutables,
`reducción funcional` es una función de transición **pura** — recuerda a la `δ` de la Máquina de Turing
(Clase 2): `(estado) -> estado nuevo`, sin mutar el estado que recibe — y `functools.reduce` la
aplica 30 veces en secuencia sobre tuplas inmutables.

También agregamos una banda de incertidumbre aproximada, construida a partir de la dispersión de
errores observada en el walk-forward — el "cono de confianza" que el naive/lambda/Ventana Móvil no muestran
por sí solos, y que es más honesto que dar sólo un número puntual sabiendo que la serie es un
paseo aleatorio.

In [ ]:
stat_lambda, p_lambda = wilcoxon(res["naive_mape"], res["lambda_mape"])
stat_movil, p_movil = wilcoxon(res["naive_mape"], res["movil_mape"])

print(f"Wilcoxon (naive vs. función lambda) -> p-valor = {p_lambda:.4f}")
print(f"Wilcoxon (naive vs. Ventana Móvil)  -> p-valor = {p_movil:.4f}")
print("\n(p < 0.05 sugiere que la diferencia frente al naive no es casualidad de muestreo;)")



## 7. Cómo usar los modelos con nuevos `t`

- `f_lambda_full = fit_lambda_function(t_all, y_all).value` → función cerrada sobre los
  parámetros ya ajustados; acepta escalar o array, dentro de `[0, 364]` para **interpolar** o
  por encima para **extrapolar**.
- `predict_ventana_movil(pasos, modelo, hist_precio, hist_fecha)` extiende el pronóstico
  recursivo cuantos días se necesiten — sin argumentos mutables por default.

In [ ]:
f_lambda_full = fit_full.value
print("Interpolación (día 100):", f_lambda_full(100))
print("Extrapolación a 45 días con Ventana Móvil:",
      predict_ventana_movil(45, y_all)[-1])



## 8. Conclusiones y limitaciones

- La serie de precios de BTC se comporta, estadísticamente, como un **paseo aleatorio**: el
  precio en sí no es estacionario, aunque sus retornos sí lo son. Por eso **ningún modelo
  determinístico puede prometer un error bajo sostenido en el tiempo** — ni éste, ni uno más
  sofisticado.
- La **función lambda** (tanh + Fourier) da un ajuste muy bueno *dentro* de la muestra, pero su
  error se dispara al extrapolar out-of-sample — información valiosa sobre los límites de un
  enfoque puramente funcional del tiempo, no un defecto del método.
- El modelo de **Ventana Móvil**, evaluado con walk-forward y contrastado con un test de Wilcoxon pareado
  (no sólo comparando promedios), muestra si existe o no señal de corto plazo explotable más
  allá del naive — con el margen de mejora moderado que es esperable en un mercado líquido.
- **Recomendación práctica:** para horizontes cortos, el Ventana Móvil walk-forward es el más confiable de
  los dos. Para entender la *forma* general de un ciclo ya ocurrido, la función lambda es más
  útil. Ninguno de los dos debería usarse como única base para decisiones financieras reales —
  esto es un ejercicio de análisis de series de tiempo, no asesoramiento de inversión.

### Nota metodológica: de dónde sale cada decisión de diseño

| Decisión en el notebook | Concepto de la cátedra |
|---|---|
| `WFConfig`, `LambdaConfig` como `@dataclass(frozen=True)` | Inmutabilidad (Clase 4) |
| `folds`, `FEATURE_COLS`, estado de `reducción funcional` como tuplas | Inmutabilidad — "se crean versiones nuevas" (Clase 4) |
| `make_model(period_u, harmonics)` devolviendo un closure | Funciones de orden superior (Clases 3-4) |
| `FitOutcome` con `.map()` que preserva la forma | Functor (Clase 4) — y anticipo de Mónada Maybe/Either (Clase 8) |
| `reduce` para acumular `FoldResult` en una tupla | Monoid: neutro `()`, combinar = concatenar (Clase 4) |
| `reducción funcional` como función `(estado) -> estado nuevo` | Función de transición pura, en la línea de la `δ` de Turing (Clase 2) |
| Modelo y error de la función lambda definidos sin mutar nada externo | Pureza y transparencia referencial (Clase 4) |
| `predict_ventana_movil` sin argumentos mutables por default | Evitar el bug clásico de Python visto en Clase 4 |

### Posibles extensiones (fuera de alcance de lo visto hasta ahora)
- Formalizar `FitOutcome` como una Mónada completa (`bind`/`flatMap`) una vez visto Maybe/Either
  en la Clase 8.
- Detección de período dominante vía FFT en vez de una lista fija de candidatos.
- Walk-forward con múltiples horizontes (1, 14, 30 días) para ver si la ventaja del Ventana Móvil se
  sostiene a distintos plazos.